In [9]:
import os
import re
import ast
import time
import pandas as pd
import requests
from dotenv import load_dotenv

# Load API keys from root .env
load_dotenv(dotenv_path="../../../.env")
if not os.getenv("OPENROUTER_API_KEY"):
    load_dotenv()

# Store all available keys in a list
api_keys = []
key1 = os.getenv("OPENROUTER_API_KEY")
key2 = os.getenv("OPENROUTER_API_KEY_NEW")

if key1: api_keys.append(key1)
if key2: api_keys.append(key2)

if not api_keys:
    raise ValueError("No API keys found! Please verify your .env file.")

# Set evaluation model to Llama 3.1 70B
EVAL_MODEL = "meta-llama/llama-3.1-70b-instruct"

print(f"Environment ready. Loaded {len(api_keys)} API keys.")
print(f"Evaluation Model set to: {EVAL_MODEL}")

Environment ready. Loaded 2 API keys.
Evaluation Model set to: meta-llama/llama-3.1-70b-instruct


In [10]:
INPUT_FILE = "gpt5.6_luna_health_condition_full_dataset.csv"
OUTPUT_FILE = "gpt5.6_luna_full_evaluated_by_llama_3_1_70b.csv"
BACKUP_FILE = "backup_" + OUTPUT_FILE

print(f"Loading full generated dataset from {INPUT_FILE}...")
df = pd.read_csv(INPUT_FILE)

def extract_target_condition(row):
    """Extracts target health condition from available columns."""
    if "target_health_condition" in row and pd.notna(row["target_health_condition"]):
        return str(row["target_health_condition"]).strip()
    val = row.get("Health Condition")
    if pd.isna(val): return None
    val_str = str(val).strip()
    if val_str.startswith("[") and val_str.endswith("]"):
        try:
            parsed = ast.literal_eval(val_str)
            if isinstance(parsed, list) and len(parsed) > 0:
                return str(parsed[0]).strip()
        except Exception:
            pass
    return val_str

df["eval_target_condition"] = df.apply(extract_target_condition, axis=1)
df = df.dropna(subset=["text", "modified_sentence", "eval_target_condition"]).reset_index(drop=True)
print(f"Loaded {len(df)} total rows.")

Loading full generated dataset from gpt5.6_luna_health_condition_full_dataset.csv...
Loaded 577 total rows.


In [11]:
# Global variable to track which key we are currently using
active_key_index = 0

PROMPT_TEMPLATE = """
You are a medical verification assistant.

Determine whether the extracted medical entity is hallucinated with respect to the given text.

Text:
{text}

Extracted Entity:
{target_entity}

If the extracted entity is hallucinated, output:
ANSWER: 1

Otherwise, output:
ANSWER: 0
"""

def evaluate_entity(text, target_entity, model_name=EVAL_MODEL):
    global active_key_index
    
    prompt = PROMPT_TEMPLATE.format(
        text=str(text).strip(),
        target_entity=str(target_entity).strip()
    )

    payload = {
        "model": model_name,
        "messages": [{"role": "user", "content": prompt}],
        "temperature": 0.0
    }

    # Loop allows us to retry if the current key is out of credits
    for _ in range(len(api_keys)):
        current_key = api_keys[active_key_index]
        
        headers = {
            "Authorization": f"Bearer {current_key}",
            "Content-Type": "application/json"
        }

        response = requests.post(
            "https://openrouter.ai/api/v1/chat/completions",
            headers=headers,
            json=payload,
            timeout=120
        )
        
        # 402 = Payment Required / 429 = Rate Limited
        if response.status_code in [402, 429]:
            print(f"   [!] Key {active_key_index + 1} hit a limit (Status {response.status_code}). Swapping keys...")
            active_key_index = (active_key_index + 1) % len(api_keys)
            continue  # Instantly retry the request with the new key
            
        response.raise_for_status()
        result = response.json()
        
        if "choices" not in result:
            raise ValueError(f"OpenRouter did not return choices: {result}")

        content = result["choices"][0]["message"]["content"].strip()
        match = re.search(r"ANSWER:\s*([01])", content, re.IGNORECASE)
        prediction = int(match.group(1)) if match else None

        return prediction, content

    raise Exception("CRITICAL ERROR: ALL API keys have reached their limits!")

In [12]:
# 1. LOAD THE BACKUP DATA & CLEAN ERRORS
if os.path.exists(BACKUP_FILE):
    print(f"Found backup file: {BACKUP_FILE}")
    df_backup = pd.read_csv(BACKUP_FILE)
    
    # Identify where the try/except block swallowed network errors
    error_pattern = "HTTPSConnectionPool|Connection aborted|ConnectionResetError|Max retries exceeded"
    bad_rows = df_backup[
        df_backup['correct_raw_response'].astype(str).str.contains(error_pattern, regex=True) |
        df_backup['hallucinated_raw_response'].astype(str).str.contains(error_pattern, regex=True)
    ]
    
    if not bad_rows.empty:
        first_bad_index = bad_rows.index[0]
        print(f"   [!] Detected network errors saved in CSV starting at row {first_bad_index + 1}.")
        print(f"   [!] Truncating corrupted rows to restore true valid progress...")
        df_backup = df_backup.iloc[:first_bad_index].copy()
        
        # Overwrite the corrupted backup file with the clean one
        df_backup.to_csv(BACKUP_FILE, index=False, encoding="utf-8-sig")

    start_index = len(df_backup)
    
    # Restore the existing lists
    correct_predictions = df_backup["correct_prediction"].tolist()
    correct_raw = df_backup["correct_raw_response"].tolist()
    hall_predictions = df_backup["hallucinated_prediction"].tolist()
    hall_raw = df_backup["hallucinated_raw_response"].tolist()
    
    # Recalculate matches so far
    correct_matches = sum(1 for p in correct_predictions if p == 0.0)
    hall_matches = sum(1 for p in hall_predictions if p == 1.0)
    
    print(f"Successfully loaded {start_index} valid rows. Resuming from row {start_index + 1}...\n")
else:
    print("No backup file found. Starting from row 1...\n")
    start_index = 0
    correct_predictions, hall_predictions = [], []
    correct_raw, hall_raw = [], []
    correct_matches, hall_matches = 0, 0

# 2. RUN THE LOOP FROM THE RESUME POINT
print(f"Resuming evaluation loop using {EVAL_MODEL}...\n")

for i in range(start_index, len(df)):
    row = df.iloc[i]
    orig_text = row["text"]
    mod_text = row["modified_sentence"]
    target_condition = row["eval_target_condition"]

    # Evaluate Original Sentence (Expected ANSWER: 0)
    try:
        pred_orig, raw_orig = evaluate_entity(orig_text, target_condition)
    except Exception as e:
        print(f"Row {i+1} [Original] Error: {e}")
        pred_orig, raw_orig = None, str(e)

    correct_predictions.append(pred_orig)
    correct_raw.append(raw_orig)
    if pred_orig == 0:
        correct_matches += 1

    # Evaluate Hallucinated Sentence (Expected ANSWER: 1)
    try:
        pred_hall, raw_hall = evaluate_entity(mod_text, target_condition)
    except Exception as e:
        print(f"Row {i+1} [Hallucinated] Error: {e}")
        pred_hall, raw_hall = None, str(e)

    hall_predictions.append(pred_hall)
    hall_raw.append(raw_hall)
    if pred_hall == 1:
        hall_matches += 1

    print(f"[{i+1}/{len(df)}] Orig Pred: {pred_orig} (Exp: 0) | Hall Pred: {pred_hall} (Exp: 1)")
    
    # AUTO-SAVE LOGIC
    if (i + 1) % 50 == 0:
        df_temp = df.loc[:i].copy()
        df_temp["correct_prediction"] = correct_predictions
        df_temp["correct_raw_response"] = correct_raw
        df_temp["hallucinated_prediction"] = hall_predictions
        df_temp["hallucinated_raw_response"] = hall_raw
        df_temp.to_csv(BACKUP_FILE, index=False, encoding="utf-8-sig")
        print(f"   --- Auto-saved backup at row {i+1} ---")

    time.sleep(0.5)

# 3. FINAL SAVE
df["correct_prediction"] = correct_predictions
df["correct_raw_response"] = correct_raw
df["hallucinated_prediction"] = hall_predictions
df["hallucinated_raw_response"] = hall_raw

df.to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig")

total_rows = len(df)
correct_accuracy = (correct_matches / total_rows) * 100 if total_rows > 0 else 0
hall_accuracy = (hall_matches / total_rows) * 100 if total_rows > 0 else 0

print("\n==============================")
print(f"Evaluator Model                : {EVAL_MODEL}")
print(f"Correct Sentence Accuracy      : {correct_accuracy:.2f}%")
print(f"Hallucinated Sentence Accuracy : {hall_accuracy:.2f}%")
print(f"Saved detailed results to      : {OUTPUT_FILE}")
print("==============================")

Found backup file: backup_gpt5.6_luna_full_evaluated_by_llama_3_1_70b.csv
   [!] Detected network errors saved in CSV starting at row 359.
   [!] Truncating corrupted rows to restore true valid progress...
Successfully loaded 358 valid rows. Resuming from row 359...

Resuming evaluation loop using meta-llama/llama-3.1-70b-instruct...

[359/577] Orig Pred: 0 (Exp: 0) | Hall Pred: 0 (Exp: 1)
[360/577] Orig Pred: 0 (Exp: 0) | Hall Pred: 1 (Exp: 1)
[361/577] Orig Pred: 0 (Exp: 0) | Hall Pred: 0 (Exp: 1)
[362/577] Orig Pred: 0 (Exp: 0) | Hall Pred: 0 (Exp: 1)
[363/577] Orig Pred: 0 (Exp: 0) | Hall Pred: 0 (Exp: 1)
[364/577] Orig Pred: 0 (Exp: 0) | Hall Pred: 1 (Exp: 1)
[365/577] Orig Pred: 0 (Exp: 0) | Hall Pred: 0 (Exp: 1)
[366/577] Orig Pred: 0 (Exp: 0) | Hall Pred: 1 (Exp: 1)
[367/577] Orig Pred: 0 (Exp: 0) | Hall Pred: 0 (Exp: 1)
[368/577] Orig Pred: 0 (Exp: 0) | Hall Pred: 0 (Exp: 1)
[369/577] Orig Pred: 0 (Exp: 0) | Hall Pred: 0 (Exp: 1)
[370/577] Orig Pred: 0 (Exp: 0) | Hall Pred: 0 